# Lekcja 2.5 — eksperymenty: porównanie strategii

Uruchamiaj komórki po kolei. Przed każdym eksperymentem zapisz swoje przewidywanie w komórce „Twoje przewidywanie” (kliknij ją dwukrotnie, żeby edytować), a dopiero potem uruchom kod.

Pierwsza komórka wybiera implementacje:

- `"mine"` używa Twoich plików `task.py` z lekcji 2.2 (BFS, DFS), 2.3 (UCS) i 2.4 (greedy, A\*, `manhattan`); najpierw rozwiąż tamte zadania;
- `"reference"` używa implementacji wzorcowych kursu. Wpisz `IMPLEMENTATION = "reference"`, jeśli któraś z Twoich implementacji jeszcze nie działa.

In [ ]:
import os
from pathlib import Path

from introai.testing.impl import load_module

# "mine" uses your task.py files from lessons 2.2-2.4; "reference" uses the course's versions.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    lessons = Path.cwd().parent
    lesson_2_2 = load_module(lessons / "02_bfs_dfs" / "task.py")
    lesson_2_3 = load_module(lessons / "03_uniform_cost_search" / "task.py")
    lesson_2_4 = load_module(lessons / "04_heuristics_and_a_star" / "task.py")
    bfs, dfs = lesson_2_2.bfs, lesson_2_2.dfs
    ucs = lesson_2_3.ucs
    greedy_best_first, a_star = lesson_2_4.greedy_best_first, lesson_2_4.a_star
    manhattan = lesson_2_4.manhattan
else:
    from introai.search.reference import a_star, bfs, dfs, greedy_best_first, ucs
    from introai.world.reference import manhattan

Funkcja `compare(warehouse)` uruchamia pięć algorytmów na jednej mapie i zwraca dla każdego liczbę kroków trasy, jej koszt, liczbę rozwinięć i czas (medianę z pięciu uruchomień). Greedy i A\* dostają odległość Manhattan do rampy. Na rysunkach niebieskie pola to pola rozwinięte, pomarańczowe to trasa, a zielone, kreskowane pola to strefy wolnego ruchu.

In [ ]:
import statistics
import time

import matplotlib.pyplot as plt

from introai.trace import TraceLimitExceeded, record
from introai.world.plot import plot_map
from introai.world.warehouse import RouteProblem, WarehouseMap, render_text

MAPS = Path.cwd().parents[2] / "data" / "maps"
LESSON = {"bfs": "2.2", "dfs": "2.2", "ucs": "2.3", "greedy_best_first": "2.4", "a_star": "2.4"}
HINT = 'or set IMPLEMENTATION = "reference" in the first cell'


def run(name, search, problem, h=None, limit=100_000):
    """Run one search; return the goal node and the expanded nodes."""
    try:
        with record(limit=limit, kinds={"expand"}) as trace:
            node = search(problem) if h is None else search(problem, h)
    except NotImplementedError:
        raise RuntimeError(
            f"{name} is not written yet - finish lesson {LESSON[name]} ({HINT})"
        ) from None
    except TraceLimitExceeded:
        raise RuntimeError(
            f"{name} expanded more than {limit} nodes - check lesson {LESSON[name]}"
        ) from None
    except TypeError as error:
        if "not supported between instances" not in str(error):
            raise
        raise RuntimeError(
            f"{name} compared two Node objects - add a counter on the heap"
        ) from None
    if node is None:
        raise RuntimeError(f"{name} found no route, but there is one - check lesson {LESSON[name]}")
    return node, [event.data["node"] for event in trace.of_kind("expand")]


def searches(warehouse):
    h = lambda cell: manhattan(cell, warehouse.dock)  # noqa: E731
    return [
        ("bfs", bfs, None),
        ("dfs", dfs, None),
        ("ucs", ucs, None),
        ("greedy_best_first", greedy_best_first, h),
        ("a_star", a_star, h),
    ]


def compare(warehouse):
    """Steps, cost, expansions and median time (ms) of the five searches on a map."""
    results = {}
    for name, search, h in searches(warehouse):
        node, expanded = run(name, search, RouteProblem(warehouse), h)
        times = []
        for _ in range(5):
            start = time.perf_counter()
            search(RouteProblem(warehouse)) if h is None else search(RouteProblem(warehouse), h)
            times.append((time.perf_counter() - start) * 1000)
        results[name] = (node, expanded, statistics.median(times))
    print(f"{'algorithm':>18} {'steps':>6} {'cost':>5} {'expanded':>9} {'ms':>6}")
    for name, (node, expanded, ms) in results.items():
        print(
            f"{name:>18} {len(node.solution()):>6} {node.path_cost:>5g} "
            f"{len(expanded):>9} {ms:>6.2f}"
        )
    return results

## Eksperyment 1: pięć algorytmów na mapach z kosztami

Mapę `congested.txt` znasz z lekcji 2.3 i 2.4. Mapa `depot.txt` to większy magazyn z dwiema zatłoczonymi poprzecznymi alejkami.

In [ ]:
congested = WarehouseMap.from_file(MAPS / "congested.txt")
depot = WarehouseMap.from_file(MAPS / "depot.txt")
print(render_text(depot))

**Twoje przewidywanie.** Na mapie `depot.txt`: które algorytmy znajdą najtańszą trasę? Który rozwinie najmniej pól? Który znajdzie najdłuższą trasę?

*Moje przewidywanie:* …

In [ ]:
print("congested.txt")
compare(congested)
print()
print("depot.txt")
depot_results = compare(depot)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10, 17))
for ax, (name, (node, expanded, _)) in zip(axes, depot_results.items()):
    title = f"{name}: {len(node.solution())} steps, cost {node.path_cost:g}"
    plot_map(depot, path=node.states(), expanded=[n.state for n in expanded], title=title, ax=ax)
fig.tight_layout()

**Do przemyślenia.** Porównaj wyniki z tabelą własności z części A. Czy na tych mapach któryś algorytm zachował się „lepiej”, niż wynikałoby z tabeli, na przykład DFS znalazł stosunkowo tanią trasę? Dlaczego to nie zmienia odpowiedzi w tabeli? Ile pól oszczędza A\* względem UCS na każdej z map i z czego wynika różnica?

## Eksperyment 2: pamięć

Pamięć mierzymy największym rozmiarem frontier. Twój kod tego nie zgłasza, więc w tym eksperymencie zawsze używamy wersji wzorcowych kursu, które raportują rozmiar frontier po każdym kroku.

**Twoje przewidywanie.** Który algorytm ma większy frontier, BFS czy DFS? Czy na każdej mapie tak samo?

*Moje przewidywanie:* …

In [ ]:
from introai.search import reference
from introai.testing.search import max_frontier

names = ["aisles", "open_floor", "congested", "depot"]
largest = {"BFS": [], "DFS": []}
for name in names:
    warehouse = WarehouseMap.from_file(MAPS / f"{name}.txt")
    for label, search in [("BFS", reference.bfs), ("DFS", reference.dfs)]:
        with record(kinds={"frontier"}) as trace:
            search(RouteProblem(warehouse))
        largest[label].append(max_frontier(trace))
    print(f"{name:>11}: largest frontier BFS {largest['BFS'][-1]:>3}, DFS {largest['DFS'][-1]:>3}")

fig, ax = plt.subplots(figsize=(8, 4))
width = 0.38
for offset, (label, color) in zip(
    (-width / 2, width / 2), [("BFS", "#2a78d6"), ("DFS", "#eb6834")]
):
    positions = [i + offset for i in range(len(names))]
    ax.bar(positions, largest[label], width=width - 0.04, color=color, label=label)
ax.set_xticks(range(len(names)), names)
ax.set_title("Largest frontier during the search", loc="left", color="#0b0b0b")
ax.set_ylabel("nodes in the frontier", color="#52514e")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
ax.legend(frameon=False, labelcolor="#0b0b0b")
fig.tight_layout()

**Do przemyślenia.** Na której mapie DFS ma największy frontier i dlaczego właśnie tam? Spójrz na długość trasy DFS na tej mapie. Co mówi wzór O(*b*·*m*), gdy *m* jest duże?

## Eksperyment 3: gdzie kończy się przeszukiwanie dokładne

Wracamy do problemu zbierania paczek z lekcji 1.3 (wersja wzorcowa kursu). A\* szuka najkrótszego planu zebrania 1, 2, …, 6 paczek na mapie z alejkami, raz z *h* = 0, a raz z heurystyką „liczba paczek, które jeszcze leżą”.

**Twoje przewidywanie.** Ile razy rośnie liczba rozwinięć z każdą kolejną paczką? Czy prosta heurystyka coś tu zmienia?

*Moje przewidywanie:* …

In [ ]:
from introai.world.reference import ParcelPickupProblem

aisles = WarehouseMap.from_file(MAPS / "aisles.txt")
CELLS = [(3, 1), (5, 4), (9, 21), (13, 15), (1, 20), (7, 13)]
counts = list(range(1, len(CELLS) + 1))
work = {"h = 0": [], "parcels left": []}
for k in counts:
    problem = ParcelPickupProblem(aisles.with_parcels(CELLS[:k]))
    for label, h in [("h = 0", lambda state: 0), ("parcels left", lambda state: len(state[1]))]:
        node, expanded = run("a_star", a_star, problem, h, limit=500_000)
        work[label].append(len(expanded))
    print(
        f"{k} parcels: plan {len(node.solution())} actions, expanded "
        f"{work['h = 0'][-1]} (h = 0) and {work['parcels left'][-1]} (parcels left)"
    )

fig, ax = plt.subplots(figsize=(7, 4))
for label, color in [("h = 0", "#2a78d6"), ("parcels left", "#eb6834")]:
    ax.plot(
        counts,
        work[label],
        color=color,
        linewidth=2,
        marker="o",
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
        label=label,
    )
ax.set_yscale("log")
ax.set_title("Nodes expanded by A* to collect the parcels", loc="left", color="#0b0b0b")
ax.set_xlabel("number of parcels", color="#52514e")
ax.set_ylabel("nodes expanded (log scale)", color="#52514e")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
ax.legend(frameon=False, labelcolor="#0b0b0b")
fig.tight_layout()

**Do przemyślenia.** Wykres ma skalę logarytmiczną. Ile rozwinięć potrzebowałby A\* przy 20 paczkach, jeśli wzrost utrzyma tempo? Dlaczego heurystyka „liczba paczek” prawie nic nie daje? Lepsze heurystyki zaprojektujesz w lekcji 2.7, a metody na naprawdę duże problemy poznasz w module 4.

## Podsumowanie

Wróć do pytań kontrolnych w lekcji. Po własnej próbie możesz porównać swoje wyniki i wnioski z odpowiedziami w `solutions/02_search/05_comparing_strategies/answers.pl.md`.